# 记录和元组

{{ video_embed | replace("%%VID%%", "E3d1IFMU558")}}

单链表是很好的数据结构，但如果你想要固定数量的元素而不是无限个呢？或者想要元素有不同类型呢？或者想通过名称而不是位置来访问元素呢？列表在这些方面都不够方便。因此，OCaml 程序员使用记录和元组。

## 记录

*记录*是多个不同类型数据的组合，每个数据都有一个名称。OCaml 记录很像 C 中的结构体。下面是一个记录类型定义示例，定义了一个宝可梦（Pokémon）类型 `mon`，重用了 [变体](variants) 部分中定义的 `ptype`：

In [ ]:
type ptype = TNormal | TFire | TWater
type mon = {name : string; hp : int; ptype : ptype}

这个类型定义了一个记录，有三个叫做 `name`、`hp`（生命值）和 `ptype` 的*字段*。每个字段的类型也被指定了。注意 `ptype` 既可以作为类型名称，也可以作为字段名称使用；这两个的*命名空间*在 OCaml 中是分开的。

为了构造一个记录类型的值，我们写一个记录表达式，看起来像这样：

In [ ]:
{name = "Charmander"; hp = 39; ptype = TFire}

所以在类型定义中，我们在字段名称和类型之间写冒号，但在表达式中我们写等号。

要访问记录并从中获取字段，我们使用你在许多其他语言中期望的点记号。例如：

In [ ]:
let c = {name = "Charmander"; hp = 39; ptype = TFire};;
c.hp

你也可以使用模式匹配来访问记录字段：

In [ ]:
match c with {name = n; hp = h; ptype = t} -> h

这里的 `n`、`h` 和 `t` 是模式变量。有一个语法糖可用，当你想对字段名和模式变量使用相同的名称时：

In [ ]:
match c with {name; hp; ptype} -> hp

这里，模式 `{name; hp; ptype}` 是 `{name = name; hp = hp; ptype = ptype}` 的语法糖。在每个子表达式中，等号左边出现的标识符是字段名，右边出现的标识符是模式变量。

{{ video_embed | replace("%%VID%%", "gXlWHvEoIvg")}}

**语法。**

记录表达式写成：

```ocaml
{f1 = e1; ...; fn = en}
```

记录表达式中 `fi=ei` 的顺序无关紧要。例如，`{f = e1; g = e2}` 完全等同于 `{g = e2; f = e1}`。

字段访问的写法是：
```ocaml
e.f
```
其中 `f` 必须是字段名称的标识符，而不是表达式。那
限制与具有类似限制的任何其他语言相同
特征---&mdash;例如，Java字段名称。如果你真的想要
*计算*要访问哪个标识符，那么实际上你想要不同的数据
结构：*map*（也有许多其他名称：*dictionary* 或
*关联列表*或*哈希表*等，尽管有细微的差别
每个条款均暗示。）

**动态语义。**

* 如果对于 `1..n` 中的所有 `i`，它保持 `ei ==> vi`，则
`{f1 = e1; ...; fn = en} ==> {f1 = v1; ...; fn = vn}`。

* 如果 `e ==> {...; f = v; ...}` 则 `e.f ==> v`。

**静态语义。**

记录类型的写法为：
```ocaml
{f1 : t1; ...; fn : tn}
```
记录类型中 `fi:ti` 的顺序无关紧要。例如，
`{f : t1; g : t2}` 完全等同于 `{g:t2;f:t1}`。

请注意，必须先定义记录类型，然后才能使用它们。  这个
使 OCaml 能够比以下情况进行更好的类型推断：
记录类型可以在没有定义的情况下使用。

类型检查规则是：

* 如果对于 `1..n` 中的所有 `i`，则它成立 `ei : ti`，并且如果 `t` 定义为
`{f1 : t1; ...; fn : tn}`，然后`{f1 = e1; ...; fn = en} : t`。请注意，
  记录表达式中提供的字段集必须是完整的字段集
  定义为记录类型的一部分（但请参阅下面有关*记录副本*的内容）。

* 如果 `e : t1` 且 `t1` 定义为 `{...; f : t2; ...}`，则
  `e.f : t2`.

**记录副本。**

还提供了另一种语法来从旧记录构造新记录：
```ocaml
{e with f1 = e1; ...; fn = en}
```
这不会改变旧记录。相反，它用新的内容构建新的记录
价值观。 `with` 之后提供的字段集不必是完整的
定义为记录类型一部分的字段集。在新复制的记录中，
未作为 `with` 的一部分提供的任何字段均从旧记录中复制。

记录副本是语法糖。相当于写
```ocaml
{ f1 = e1;   ...; fn = en;
  g1 = e.g1; ...; gn = e.gn }
```
其中 `gi` 的集合是记录类型的所有字段减去
`fi` 组。

**模式匹配。**

我们将以下新模式形式添加到合法模式列表中：

* `{f1 = p1; ...; fn = pn}`

我们扩展了模式何时匹配一个值并产生一个的定义
绑定如下：

* 如果对于 `1..n` 中的所有 `i`，则认为 `pi` 与 `vi` 匹配并产生
绑定 $b_i$，则记录模式 `{f1 = p1; ...; fn = pn}` 与
  记录值 `{f1 = v1; ...; fn = vn; ...}` 并生成集合 $\bigcup_i
  b_i$ 绑定。请注意，记录值的字段数可能多于记录值的字段数。
  记录模式可以。

作为语法糖，提供了另一种形式的记录模式：
`{f1; ...; fn}`。它会被去糖为 `{f1 = f1; ...; fn = fn}`。

## 元组

与记录一样，*元组*是其他类型数据的组合。但不是
命名*组件*时，它们通过位置来标识。这是一些例子
元组：
```ocaml
(1, 2, 10)
(true, "Hello")
([1; 2; 3], (0.5, 'X'))
```
具有两个组件的元组称为“对”。具有三个组件的元组是
称为*三元组*。除此之外，我们通常只使用单词 *tuple* 而不是
继续基于数字的命名方案。

```{tip}
除了大约三个组件之外，可以说使用记录比使用记录更好
元组，因为程序员很难记住哪个组件是
应该代表什么信息。
```

元组的构建很简单：只需编写元组即可，如上所示。再次访问
涉及到模式匹配，例如：

In [ ]:
match (1, 2, 3) with (x, y, z) -> x + y + z

{{ video_embed | replace("%%VID%%", "4QNzC2KZ5I4")}}

**语法。**

写了一个元组
```ocaml
(e1, e2, ..., en)
```
括号并不完全是强制性的 &mdash; 通常你的代码可以
成功解析没有它们&mdash; 但它们通常被认为是
良好的风格包括。

**动态语义。**

* 如果对于 `1..n` 中的所有 `i` 都持有 `ei ==> vi`，则
`(e1, ..., en) ==> (v1, ..., vn)`。

**静态语义。**

元组类型是使用新的类型构造函数`*`编写的，这是不同的
比乘法运算符。类型 `t1 * ... * tn` 是元组的类型
第一个组件的类型为 `t1`，...，第 n 个组件的类型为 `tn`。

* 如果对于 `1..n` 中的所有 `i` 都持有 `ei : ti`，则
`(e1, ..., en) : t1 * ... * tn`。

**模式匹配。**

我们将以下新模式形式添加到合法模式列表中：

* `(p1, ..., pn)`

括号也不是完全强制的，但通常是惯用的
包括。

我们扩展了模式何时匹配一个值并产生一个的定义
绑定如下：

* 如果对于 `1..n` 中的所有 `i`，则认为 `pi` 与 `vi` 匹配并产生
绑定 $b_i$，则元组模式 `(p1, ..., pn)` 与元组值匹配
  `(v1, ..., vn)` 并生成绑定集 $\bigcup_i b_i$。请注意
  元组值必须具有与元组完全相同的组件数
  模式确实如此。

## 变体与元组和记录

{{ video_embed | replace("%%VID%%", "9kyOH1kpmjk")}}

{{ video_embed | replace("%%VID%%", "oMOO-cWrHuw")}}

```{note}
上面的第二个视频使用了更高级的变体示例，这些示例将
在[后面的小节](algebraic_data_types)中学习。
```

变体和我们刚刚学到的类型之间的巨大区别（记录和
元组）是变体类型的值是一组可能性中的一个，
而元组或记录类型的值提供*每个*一组
的可能性。回到我们的示例，类型 `day` 的值是**之一**
`Sun` 或 `Mon` 等。但是 `mon` 类型的值提供 **每个** `string`
以及 `int` 和 `ptype`。请注意，在前两句话中，这个词
“or”与变体类型相关联，“and”一词与
元组和记录类型。如果你想做出决定，这是一个很好的线索
是否要使用变体、元组或记录：如果你需要一件
数据*或*另一个，你想要一个变体；如果你需要一份数据*和*
另一个，你想要一个元组或记录。

其中之一类型通常称为 *sum 类型*，而each-of 类型则称为
*积类型*。这些名字来自集合论。变体就像
[disjoint union][disjun]，因为变体的每个值都来自多个值之一
底层集合（到目前为止，每个集合只是一个构造函数
因此基数为一）。不相交联盟有时确实是用
求和运算符 $\Sigma$。元组/记录就像
[Cartesian product][cartprod]，因为元组或记录的每个值都包含
来自许多基础集合中的每一个的值。笛卡尔积通常写成
使用产品运算符 $\times$ 或 $\Pi$。

[disjun]: https://en.wikipedia.org/wiki/Disjoint_union
[cartprod]: https://en.wikipedia.org/wiki/Cartesian_product